- https://datahonor.com/blog/2025/06/03/llm_kv_cache/

### mha => gqa

- 几个“查询专家”（Q 头）共享同一份参考资料（K 和 V 头）
    - num_attention_heads：这还是指“查询专家” Q 的总数量。
    - num_kv_heads：这是指 K 和 V 的头的数量。
    - 分组：将 num_attention_heads 个 Q 头分成 num_kv_heads 个组。每个组内的所有 Q 头将共享同一套 K 和 V。
    - 投影：模型仍然为每个 Q 头生成一个独立的 Q 向量。但是，模型只生成 num_kv_heads 组 K 和 V 向量。
- $X \in \mathbb{R}^{L \times d_{model}}$, $N_q$（查询头的数量 (num_attention_heads)）, $N_{kv}$: 键/值头的数量 (num_kv_heads)
    - $g = N_q / N_{kv}$: 每个 KV 头被共享的 Q 头的数量（分组大小）
    - $d_h$: 每个查询头的维度, $d_k$: 每个键/值头的维度 (通常 $d_h = d_k$)，$d_h=\frac{d_{model}}{N_q}$
    - $W_i^Q \in \mathbb{R}^{d_{model} \times d_h}$: 第 $i$ 个查询头的投影矩阵, $i \in \{1, \dots, N_q\}$
    - $W_j^K \in \mathbb{R}^{d_{model} \times d_k}$: 第 $j$ 个键头的投影矩阵, $j \in \{1, \dots, N_{kv}\}$
    - $W_j^V \in \mathbb{R}^{d_{model} \times d_k}$: 第 $j$ 个值头的投影矩阵, $j \in \{1, \dots, N_{kv}\}$
    - $W^O \in \mathbb{R}^{N_q d_h \times d_{model}}$: 输出投影矩阵

#### 计算流程

- 投影

$$
\begin{aligned}
\text{Queries: } & Q_i = X W_i^Q, & \quad \text{for } i = 1, \dots, N_q \\
\text{Keys: } & K_j = X W_j^K, & \quad \text{for } j = 1, \dots, N_{kv} \\
\text{Values: } & V_j = X W_j^V, & \quad \text{for } j = 1, \dots, N_{kv}
\end{aligned}
$$

- 分组与注意力计算 (Grouping and Attention Calculation)
    - 对于第 $i$ 个查询头 $Q_i$, 它需要与一个特定的键/值头对 $(K_j, V_j)$, 进行交互。这个对应关系由分组决定。第 $i$ 个查询头所属的组索引为 $j = \lfloor (i-1) / g \rfloor + 1$
        - 第 $i$ 个头的输出 $\text{Head}_i$ 为：

$$
\text{Head}_i = \text{Attention}(Q_i, K_j, V_j) = \text{softmax}\left(\frac{Q_i K_j^T}{\sqrt{d_k}}\right) V_j
$$

- 拼接与输出 (Concatenation and Final Projection)
    - 所有 $N_q$ 个头的输出被拼接在一起，然后通过最终的输出投影矩阵 $W^O$ 得到最终的输出。

    $$
    \begin{aligned}
    \text{MultiHeadOutput} &= \text{Concat}(\text{Head}_1, \text{Head}_2, \dots, \text{Head}_{N_q})\\
    \text{Output} &= \text{MultiHeadOutput} \cdot W^O
    \end{aligned}
    $$
    - $\text{Concat}(\cdot)$ 操作将所有 $\text{Head}_i \in \mathbb{R}^{L \times d_h}$ 拼接成一个大的矩阵 $\text{MultiHeadOutput} \in \mathbb{R}^{L \times (N_q d_h)}$

*   MHA (Multi-Head Attention): 当 N_kv = N_q 时，分组大小 g=1。每个查询头 Q_i 都对应唯一的键/值对 (K_i, V_i)。这退化为标准的多头注意力机制。
*   MQA (Multi-Query Attention): 当 N_kv = 1 时，分组大小 g=N_q。所有 N_q 个查询头 Q_i 都共享同一对键/值 (K_1, V_1)。这是最高效的 KV 缓存共享模式。
*   GQA (Grouped-Query Attention): 当 1 < N_kv < N_q 时，就是分组查询注意力。它在模型性能和推理效率之间提供了一个灵活的权衡。